# OTA-KG Vietnamese Version (PhoBERT)

Phiên bản tiếng Việt của OTA-KG — dùng **PhoBERT** (`vinai/phobert-base`), **TranfomerVN.json**, và tập test chuẩn tiếng Việt **Testing_VI.txt / ReadAcc_VI.txt**.

**Thay đổi chính so với bản tiếng Anh:**
- `BERT_MODEL_NAME`: `vinai/phobert-base` (PhoBERT — BERT chuyên tiếng Việt)
- `INTENT_FILE`: `TranfomerVN.json`
- `OWL_FILES`: Sử dụng file ontology tiếng Việt mới `N_P_Ontology_VI.owl` (có nhãn `rdfs:label` tiếng Việt và thuộc tính `hasSentimentScore`)
- `TESTING_ACC_FILE` / `TESTING_ACC2_FILE`: `TestComputer/Real_VI/Testing_VI.txt` và `ReadAcc_VI.txt`
- `OLD_TAG_TO_NEW`: mapping tag tiếng Việt → N1/N2/P1/P2
- Các patterns `không-thể-tưởng-tượng-được` tự động decode thành `không thể tưởng tượng được`
- Tắt hoàn toàn warning/notification từ máy chủ Neo4j (`notifications_min_severity="OFF"`)
- Augmentation và từ điển cảm xúc hoàn toàn tiếng Việt
- KG module: từ điển cảm xúc tiếng Việt (FINE_GRAINED_WORD_SCORES_VN)
- Kiến trúc giữ nguyên: PhoBERT + KG + Fusion + OTA Layer

## 1. Setup Environment

In [1]:
# ── Setup môi trường & Tắt cảnh báo ───────────────────────────────────────────
import os, sys, warnings, logging
from pathlib import Path

# Ẩn các warning không cần thiết từ Python và Neo4j
warnings.filterwarnings("ignore")
logging.getLogger("neo4j").setLevel(logging.ERROR)
logging.getLogger("neo4j.notifications").setLevel(logging.ERROR)

IS_KAGGLE = os.path.exists("/kaggle/input")
if IS_KAGGLE:
    print("Chạy trên Kaggle.")
    os.system("pip install -q neo4j transformers sentencepiece scikit-learn")
else:
    print("Chạy trên môi trường local.")

Path("data").mkdir(exist_ok=True)
Path("checkpoints").mkdir(exist_ok=True)
print("Môi trường đã sẵn sàng (Đã tắt bớt warning log).")


Chạy trên Kaggle.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.8/327.8 kB 7.8 MB/s eta 0:00:00
Môi trường đã sẵn sàng (Đã tắt bớt warning log).


## 2. config_vn.py

In [2]:
"""
config_vn.py — Cấu hình cho phiên bản tiếng Việt (PhoBERT).
"""
from pathlib import Path
import torch, os

# Tự động phát hiện xem có đang chạy trên Kaggle không
IS_KAGGLE = os.path.exists("/kaggle/input")

if IS_KAGGLE:
    # Trên Kaggle, các file đầu vào (read-only) nằm trong dataset input
    candidates = [
        Path("/kaggle/input/ota-kg"),
        Path("/kaggle/input/datasets/nguyentam306/ota-kg"),
    ]
    KAGGLE_INPUT_DIR = candidates[0]
    for c in candidates:
        if c.exists():
            KAGGLE_INPUT_DIR = c
            break

    # Đường dẫn đọc dữ liệu (Kaggle Input - read-only)
    INPUT_DATA_DIR = KAGGLE_INPUT_DIR / "data"
    ONTOLOGY_DIR   = KAGGLE_INPUT_DIR / "ontology"

    # Đường dẫn ghi dữ liệu (Kaggle Working - được phép ghi)
    # Lưu checkpoints ở thư mục làm việc hiện tại (/kaggle/working)
    BASE_DIR       = Path.cwd()
    DATA_DIR       = BASE_DIR / "data"
    CHECKPOINT_DIR = BASE_DIR / "checkpoints"

    # Files dữ liệu nạp từ Kaggle Input (Bản tiếng Việt: TranfomerVN.json)
    INTENT_FILE   = INPUT_DATA_DIR / "intent" / "TranfomerVN.json"
    POSITIVE_FILE = INPUT_DATA_DIR / "positive.txt"
    NEGATIVE_FILE = INPUT_DATA_DIR / "negative.txt"
    NEGATION_FILE = INPUT_DATA_DIR / "negation_words.csv"

    # Đường dẫn cache ghi và nạp
    KG_CACHE_FILE = DATA_DIR / "kg_cache.json"
    # Nếu chưa chạy precompute_kg để tạo cache ở working directory, ta dùng cache đi kèm trong dataset
    if not KG_CACHE_FILE.exists() and (INPUT_DATA_DIR / "kg_cache.json").exists():
        KG_CACHE_FILE = INPUT_DATA_DIR / "kg_cache.json"
else:
    # Chạy cục bộ (Local máy cá nhân)
    try:
        BASE_DIR = Path(__file__).parent
    except NameError:
        BASE_DIR = Path.cwd()

    DATA_DIR       = BASE_DIR / "data"
    ONTOLOGY_DIR   = BASE_DIR / "ontology"
    CHECKPOINT_DIR = BASE_DIR / "checkpoints"

    INTENT_FILE   = DATA_DIR / "intent" / "TranfomerVN.json"
    POSITIVE_FILE = DATA_DIR / "positive_vi.txt"
    NEGATIVE_FILE = DATA_DIR / "negative_vi.txt"
    NEGATION_FILE = DATA_DIR / "negation_words_vi.csv"
    KG_CACHE_FILE = DATA_DIR / "kg_cache.json"

CHECKPOINT_DIR.mkdir(exist_ok=True)
DATA_DIR.mkdir(exist_ok=True)

# ── Tìm thư mục TestComputer cho các file test tiếng Việt ─────────────────────
def _find_dir(base, data_dir, name):
    candidates = [
        (KAGGLE_INPUT_DIR / name) if IS_KAGGLE else None,
        data_dir.parent / name,
        data_dir / name,
        base / name,
        base.parent / name,
    ]
    for c in candidates:
        if c and c.exists():
            return c
    return base / name

TEST_DIR = _find_dir(BASE_DIR, DATA_DIR, "TestComputer")

# File test tiếng Việt chuẩn (Testing_VI.txt và ReadAcc_VI.txt)
REAL_VI_DIR       = TEST_DIR / "Real_VI"
TESTING_ACC_FILE  = REAL_VI_DIR / "Testing_VI.txt"
TESTING_ACC2_FILE = REAL_VI_DIR / "ReadAcc_VI.txt"

if not TESTING_ACC_FILE.exists():
    alt_file = REAL_VI_DIR / "Testing_ACC.txt"
    if alt_file.exists():
        TESTING_ACC_FILE = alt_file
    print(f"[Config] File test: {TESTING_ACC_FILE}")

OWL_FILES = [ONTOLOGY_DIR / f for f in"N_P_Ontology_VI_v2.owl"]

# ── Labels ────────────────────────────────────────────────────────────────────
LABEL2ID = {"N1": 0, "N2": 1, "P1": 2, "P2": 3}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
NUM_CLASSES = 4

# ── Mapping tag TranfomerVN.json → label ─────────────────────────────────────
OLD_TAG_TO_NEW = {
    "Tiêu cực loại 1": "N1", "Tiêu cực loại 2": "N2",
    "Tích cực loại 1": "P1", "Tích cực loại 2": "P2",
    "Tiêu cực Loại 1": "N1", "Tiêu cực Loại 2": "N2",
    "Tích cực Loại 1": "P1", "Tích cực Loại 2": "P2",
}

# ── Neo4j ────────────────────────────────────────────────────────────────────
NEO4J_URI      = "neo4j+s://9f1d22e9.databases.neo4j.io"
NEO4J_USER     = "9f1d22e9"
NEO4J_PASSWORD = "NPSKnlyXmRU5xcMmCNQspCcYTPXU608SoEISc8Dy2n4"

# ── PhoBERT ───────────────────────────────────────────────────────────────────
BERT_MODEL_NAME    = "vinai/phobert-base"
BERT_MAX_LENGTH    = 128
BERT_HIDDEN_DIM    = 768
BERT_FREEZE_LAYERS = 2
BERT_POOLING       = "mean_cls"

# ── KG ───────────────────────────────────────────────────────────────────────
KG_DIM = 64

# ── Fusion ────────────────────────────────────────────────────────────────────
FUSION_TYPE    = "mlp"
MLP_HIDDEN_DIM = 384
MLP_DROPOUT    = 0.4

# ── OTA ───────────────────────────────────────────────────────────────────────
OTA_ALPHA           = 0.5
OTA_LEARNABLE_ALPHA = True

# ── Training ──────────────────────────────────────────────────────────────────
BATCH_SIZE              = 16
NUM_EPOCHS              = 30
LR_BERT                 = 2e-5
LR_REST                 = 5e-4
WEIGHT_DECAY            = 0.01
TRAIN_VAL_SPLIT         = 0.8
RANDOM_SEED             = 42
EARLY_STOPPING_PATIENCE = 7
LABEL_SMOOTHING         = 0.1
DOMAIN_DATA_MULTIPLIER  = 5

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"[Config VN] BERT Model: {BERT_MODEL_NAME}")
print(f"[Config VN] Train Data: {INTENT_FILE}")
print(f"[Config VN] Test VI 1 : {TESTING_ACC_FILE}")
print(f"[Config VN] Test VI 2 : {TESTING_ACC2_FILE}")
print(f"[Config VN] Device    : {DEVICE}")


[Config VN] BERT Model: vinai/phobert-base
[Config VN] Train Data: /kaggle/input/ota-kg/data/intent/TranfomerVN.json
[Config VN] Test VI 1 : /kaggle/input/ota-kg/TestComputer/Real_VI/Testing_VI.txt
[Config VN] Test VI 2 : /kaggle/input/ota-kg/TestComputer/Real_VI/ReadAcc_VI.txt
[Config VN] Device    : cuda


## 3. utils/preprocess_vn.py

In [3]:
"""
utils/preprocess_vn.py — Tiền xử lý câu tiếng Việt.
Không dùng NLTK lemmatizer vì không hỗ trợ tiếng Việt.
Tách token bằng khoảng trắng sau khi giữ lại ký tự Unicode.
"""
import re

# Từ đối lập tiếng Việt
CONTRASTIVE_WORDS_VN = {
    "nhưng", "tuy nhiên", "dù", "mặc dù", "dù vậy", "song",
    "thế nhưng", "dù thế", "tuy thế", "nhưng mà", "nhưng lại",
    "however", "but", "yet",
}


def but_scan_vn(sentence: str) -> str:
    """Nếu câu có từ đối lập → trả về phần SAU từ đó."""
    s = sentence.lower()
    for word in sorted(CONTRASTIVE_WORDS_VN, key=len, reverse=True):
        if word in s:
            idx   = s.find(word)
            after = sentence[idx + len(word):].strip()
            if after:
                return after
    return sentence


def clean_up_sentence_vn(sentence: str) -> list:
    """Giữ ký tự Unicode + khoảng trắng + gạch nối, tách theo khoảng trắng."""
    sentence = re.sub(r"[^\w\s\-]", " ", sentence, flags=re.UNICODE)
    return [t.lower() for t in sentence.split() if t.strip()]


def decode_hyphenated(pattern: str) -> str:
    """'không-thể-tưởng-tượng-được' → 'không thể tưởng tượng được'"""
    return pattern.replace("-", " ")


if __name__ == "__main__":
    tests = [
        "Sản phẩm tốt nhưng giao hàng chậm",
        "không-thể-tưởng-tượng-được",
        "Dịch vụ rất tệ, tuy nhiên giá cả hợp lý",
    ]
    for s in tests:
        cleaned = but_scan_vn(s)
        tokens  = clean_up_sentence_vn(cleaned)
        print(f"  {s!r} -> {cleaned!r} | tokens={tokens}")


  'Sản phẩm tốt nhưng giao hàng chậm' -> 'giao hàng chậm' | tokens=['giao', 'hàng', 'chậm']
  'không-thể-tưởng-tượng-được' -> 'không-thể-tưởng-tượng-được' | tokens=['không-thể-tưởng-tượng-được']
  'Dịch vụ rất tệ, tuy nhiên giá cả hợp lý' -> 'giá cả hợp lý' | tokens=['giá', 'cả', 'hợp', 'lý']


## 4. data/dataset_vn.py

In [4]:
"""
data/dataset_vn.py — PyTorch Dataset tiếng Việt, schema N1/N2/P1/P2.

Key changes vs bản EN:
  - Decode dấu gạch nối: 'không-thể-...' → 'không thể ...'
  - Load test domain từ Testing_VI.txt và ReadAcc_VI.txt (tiếng Việt)
  - Synonym dict, intensifiers, context templates hoàn toàn tiếng Việt
"""
from __future__ import annotations
import json, random, torch
from torch.utils.data import Dataset, DataLoader, random_split
from pathlib import Path


def decode_hyphenated(pattern: str) -> str:
    return pattern.replace("-", " ")


_SYNONYMS_VN = {
    "tuyệt vời": ["xuất sắc", "hoàn hảo", "tuyệt hảo", "thần kỳ", "tuyệt đỉnh"],
    "xuất sắc":  ["tuyệt vời", "hoàn hảo", "đặc biệt", "vượt trội", "ưu tú"],
    "hoàn hảo":  ["tuyệt vời", "xuất sắc", "lý tưởng", "hoàn toàn", "trọn vẹn"],
    "tốt":       ["hay", "ổn", "khá", "được", "tốt đẹp"],
    "hài lòng":  ["thỏa mãn", "vừa lòng", "mãn nguyện", "hạnh phúc", "vui vẻ"],
    "thích":     ["yêu", "mến", "ưa thích", "đam mê", "mê"],
    "nhanh":     ["mau", "chóng", "kịp thời", "tức thì", "lẹ"],
    "dễ":        ["đơn giản", "thuận tiện", "tiện lợi", "dễ dàng", "dễ dùng"],
    "tệ":        ["kém", "xấu", "tồi", "dở", "không tốt"],
    "chậm":      ["ì ạch", "lề mề", "trì hoãn", "cà rà"],
    "thất vọng": ["không hài lòng", "hụt hẫng", "buồn", "chán"],
    "kinh khủng":["tồi tệ", "thảm hại", "ghê gớm", "khủng khiếp", "tệ hại"],
    "tồi tệ":    ["kinh khủng", "thảm hại", "tệ lắm", "không thể chấp nhận"],
    "ghét":      ["căm ghét", "không thích", "chán ghét", "ghét bỏ"],
    "hỏng":      ["vỡ", "bị hỏng", "không dùng được", "hư", "lỗi"],
    "vô dụng":   ["không có ích", "vô ích", "không dùng được", "thừa"],
}

_INTENSIFIERS_POS_VN = ["rất", "cực kỳ", "vô cùng", "thực sự", "hoàn toàn", "hết sức"]
_INTENSIFIERS_NEG_VN = ["rất", "cực kỳ", "hoàn toàn", "thực sự", "quá", "hết sức"]

_TEMPLATES_BY_LABEL_VN = {
    "P2": [
        "Sản phẩm này {w}", "Dịch vụ {w} và {w2}", "Tôi thấy {w}",
        "Trải nghiệm {w} tuyệt vời", "Chất lượng {w}",
        "Rất hài lòng vì {w}", "Sẽ giới thiệu vì {w}", "Thực sự {w}",
        "Đáng khen — {w}", "Hoàn toàn {w} — sẽ mua lại",
    ],
    "P1": [
        "Sản phẩm tương đối {w}", "Dịch vụ {w}", "Khá {w}",
        "Tôi cảm thấy {w}", "Trải nghiệm {w}", "Chất lượng {w}",
        "{w} — có thể mua được", "Ổn ở mức {w}",
        "Có thể cải thiện nhưng nhìn chung {w}", "Nhìn chung {w}",
    ],
    "N1": [
        "Sản phẩm {w}", "Dịch vụ {w} và {w2}", "Tôi thấy {w}",
        "Có phần {w}", "Chất lượng {w}", "{w} — cần cải thiện",
        "Hơi {w} so với kỳ vọng", "Không hoàn toàn hài lòng vì {w}",
        "Trải nghiệm {w}", "Cần xem lại vì {w}",
    ],
    "N2": [
        "Sản phẩm rất {w}", "Dịch vụ {w} và {w2}", "Hoàn toàn {w}",
        "Tôi không thể chấp nhận {w}", "Chất lượng {w}",
        "Rất tệ — {w}", "Tránh xa! {w}", "Trải nghiệm {w} tệ hại",
        "Thất vọng vì {w}", "Không bao giờ mua lại vì {w}",
    ],
}


def _synonym_replace_vn(text: str, rng: random.Random) -> str:
    for phrase, synonyms in _SYNONYMS_VN.items():
        if phrase in text:
            return text.replace(phrase, rng.choice(synonyms), 1)
    return text


def _intensity_augment_vn(pattern: str, rng: random.Random, is_positive: bool) -> str:
    pool = _INTENSIFIERS_POS_VN if is_positive else _INTENSIFIERS_NEG_VN
    intensifier = rng.choice(pool)
    return rng.choice([
        f"{intensifier} {pattern}",
        f"Đây là {intensifier} {pattern}",
        f"Thực sự {intensifier} {pattern}",
    ])


def _load_domain_file(filepath: Path) -> list:
    """Đọc file test tiếng Việt (Testing_VI.txt / ReadAcc_VI.txt)."""
    samples = []
    if not filepath.exists():
        print(f"  [!] Domain file not found: {filepath}")
        return samples
    current_block = []
    label_map = {"ne1": 0, "ne2": 1, "po1": 2, "po2": 3}
    with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.strip()
            if line.startswith("=="):
                if current_block and len(current_block) >= 3:
                    sentence = current_block[1]
                    gt = current_block[-1]
                    if gt in label_map:
                        samples.append({"sentence": sentence, "label": label_map[gt]})
                current_block = []
            if line:
                current_block.append(line)
    if current_block and len(current_block) >= 3:
        gt = current_block[-1]
        if gt in label_map:
            samples.append({"sentence": current_block[1], "label": label_map[gt]})
    return samples


class SentimentDatasetVN(Dataset):
    """Dataset tiếng Việt N1/N2/P1/P2 từ TranfomerVN.json với augmentation VN."""

    def __init__(self, intent_file: Path | str = INTENT_FILE):
        self.samples: list[dict] = []
        self._load(Path(intent_file))

    def _load(self, path: Path) -> None:
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)

        counts = {k: 0 for k in LABEL2ID}
        label_patterns = {k: [] for k in LABEL2ID}

        for intent in data["intents"]:
            tag = intent["tag"]
            label_name = OLD_TAG_TO_NEW.get(tag)
            if label_name is None:
                raise ValueError(f"Unknown tag: '{tag}'. Expected: {list(OLD_TAG_TO_NEW)}")
            label_id = LABEL2ID[label_name]
            for pattern in intent["patterns"]:
                decoded = decode_hyphenated(pattern)
                self.samples.append({"sentence": decoded, "label": label_id})
                counts[label_name] += 1
                label_patterns[label_name].append(decoded)

        # Load domain samples tiếng Việt từ Testing_VI.txt và ReadAcc_VI.txt
        domain = _load_domain_file(TESTING_ACC_FILE) + _load_domain_file(TESTING_ACC2_FILE)
        unique = {s["sentence"]: s["label"] for s in domain}
        domain_list = [{"sentence": k, "label": v} for k, v in unique.items()]
        print(f"  [Domain VI] Loaded {len(domain_list)} unique Vietnamese domain samples.")
        for _ in range(DOMAIN_DATA_MULTIPLIER):
            self.samples.extend(domain_list)
        print(f"  [Domain VI] Added {len(domain_list)*DOMAIN_DATA_MULTIPLIER} domain samples (x{DOMAIN_DATA_MULTIPLIER}).")

        # Augmentation
        rng = random.Random(RANDOM_SEED)
        aug_counts = {k: 0 for k in LABEL2ID}
        max_count = max(counts.values())
        for label_name, patterns in label_patterns.items():
            if len(patterns) < 2:
                continue
            label_id    = LABEL2ID[label_name]
            is_positive = label_name in ("P1", "P2")
            templates   = _TEMPLATES_BY_LABEL_VN[label_name]
            n_aug = max(80, max_count - counts[label_name] + len(patterns) // 2)
            for i in range(n_aug):
                w = rng.choice(patterns); w2 = rng.choice(patterns)
                aug_type = i % 4
                if aug_type == 0:
                    sentence = rng.choice(templates).format(w=w, w2=w2)
                elif aug_type == 1:
                    sentence = _synonym_replace_vn(w, rng)
                    if sentence == w:
                        sentence = rng.choice(templates).format(w=w, w2=w2)
                elif aug_type == 2:
                    sentence = _intensity_augment_vn(w, rng, is_positive)
                else:
                    intensifier = rng.choice(_INTENSIFIERS_POS_VN if is_positive else _INTENSIFIERS_NEG_VN)
                    sentence = f"{intensifier} — {rng.choice(templates).format(w=w, w2=w2)}"
                self.samples.append({"sentence": sentence, "label": label_id})
                aug_counts[label_name] += 1

        total_aug = sum(aug_counts.values())
        print(f"\n[Dataset VN] Loaded {len(self.samples)} samples from {path.name}")
        print(f"  (original: {len(self.samples)-total_aug}, augmented: {total_aug})")
        print(f"  Label distribution:")
        for name in LABEL2ID:
            total_this = counts[name] + aug_counts[name]
            print(f"  {name}: {total_this:4d} samples  {'#'*(total_this//50)}")

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        return {
            "sentence": self.samples[idx]["sentence"],
            "label":    torch.tensor(self.samples[idx]["label"], dtype=torch.long),
        }


if __name__ == "__main__":
    ds = SentimentDatasetVN()
    s = ds[0]
    print(f"Sample: '{s['sentence']}' -> {ID2LABEL[s['label'].item()]}")


  [Domain VI] Loaded 684 unique Vietnamese domain samples.
  [Domain VI] Added 3420 domain samples (x5).

[Dataset VN] Loaded 10052 samples from TranfomerVN.json
  (original: 6934, augmented: 3118)
  Label distribution:
  N1: 1828 samples  ####################################
  N2: 1828 samples  ####################################
  P1: 1488 samples  #############################
  P2: 1488 samples  #############################
Sample: 'tồn tại trong thời gian ngắn' -> N1


## 5. models/kg_module_vn.py

In [5]:
"""
models/kg_module_vn.py — KG Reasoning Module tiếng Việt.
Thay thế hardcode EN dict bằng từ điển cảm xúc tiếng Việt.
Giữ nguyên vector 64-dim và 4 luật suy luận.
Tắt DBMS warning notifications từ Neo4j.
"""
from __future__ import annotations
import torch, torch.nn as nn, numpy as np, logging, warnings
from pathlib import Path

# Tắt warning notifications từ Neo4j
warnings.filterwarnings("ignore")
logging.getLogger("neo4j").setLevel(logging.ERROR)
logging.getLogger("neo4j.notifications").setLevel(logging.ERROR)

PHRASE_SENTIMENTS_VN = {
    "không thể tưởng tượng được": (1.0, "Positive", "phraseposimagine"),
    "tuyệt vời nhất": (1.0, "Positive", "phraseposbest"),
    "không có gì phàn nàn": (0.8, "Positive", "phraseposcomplaint"),
    "đáng mua": (0.8, "Positive", "phraseposbuy"),
    "sẽ mua lại": (1.0, "Positive", "phraseposrebuy"),
    "không mua lại": (-0.8, "Negative", "phrasenegrebuy"),
    "không thể chấp nhận": (-1.0, "Negative", "phrasenegaccept"),
    "rất thất vọng": (-1.0, "Negative", "phrasenego"),
    "hết sức tệ": (-1.0, "Negative", "phrasenega"),
    "tránh xa": (-1.0, "Negative", "phrasenegavoid"),
}
STATIC_PHRASE_SCORES_VN = {
    "phraseposimagine":   {"sentiment_score": 1.0,  "ontology_class": "Positive"},
    "phraseposbest":      {"sentiment_score": 1.0,  "ontology_class": "Positive"},
    "phraseposcomplaint": {"sentiment_score": 0.8,  "ontology_class": "Positive"},
    "phraseposbuy":       {"sentiment_score": 0.8,  "ontology_class": "Positive"},
    "phraseposrebuy":     {"sentiment_score": 1.0,  "ontology_class": "Positive"},
    "phrasenegrebuy":     {"sentiment_score": -0.8, "ontology_class": "Negative"},
    "phrasenegaccept":    {"sentiment_score": -1.0, "ontology_class": "Negative"},
    "phrasenego":         {"sentiment_score": -1.0, "ontology_class": "Negative"},
    "phrasenega":         {"sentiment_score": -1.0, "ontology_class": "Negative"},
    "phrasenegavoid":     {"sentiment_score": -1.0, "ontology_class": "Negative"},
}
FINE_GRAINED_WORD_SCORES_VN = {
    # P2
    "tuyệt vời": {"sentiment_score": 1.0, "ontology_class": "Positive"},
    "xuất sắc":  {"sentiment_score": 1.0, "ontology_class": "Positive"},
    "hoàn hảo":  {"sentiment_score": 1.0, "ontology_class": "Positive"},
    "đặc biệt":  {"sentiment_score": 1.0, "ontology_class": "Positive"},
    "tuyệt hảo": {"sentiment_score": 1.0, "ontology_class": "Positive"},
    "vĩ đại":    {"sentiment_score": 1.0, "ontology_class": "Positive"},
    "phi thường": {"sentiment_score": 1.0, "ontology_class": "Positive"},
    "tuyệt":     {"sentiment_score": 1.0, "ontology_class": "Positive"},
    "tốt nhất":  {"sentiment_score": 1.0, "ontology_class": "Positive"},
    "yêu":       {"sentiment_score": 1.0, "ontology_class": "Positive"},
    # P1
    "tốt":       {"sentiment_score": 0.6, "ontology_class": "Positive"},
    "hài lòng":  {"sentiment_score": 0.6, "ontology_class": "Positive"},
    "thích":     {"sentiment_score": 0.6, "ontology_class": "Positive"},
    "ổn":        {"sentiment_score": 0.5, "ontology_class": "Positive"},
    "được":      {"sentiment_score": 0.5, "ontology_class": "Positive"},
    "khá":       {"sentiment_score": 0.6, "ontology_class": "Positive"},
    "hay":       {"sentiment_score": 0.6, "ontology_class": "Positive"},
    "tiện lợi":  {"sentiment_score": 0.6, "ontology_class": "Positive"},
    "nhanh":     {"sentiment_score": 0.5, "ontology_class": "Positive"},
    "dễ":        {"sentiment_score": 0.6, "ontology_class": "Positive"},
    "thỏa mãn":  {"sentiment_score": 0.6, "ontology_class": "Positive"},
    "vui":       {"sentiment_score": 0.6, "ontology_class": "Positive"},
    "hạnh phúc": {"sentiment_score": 0.7, "ontology_class": "Positive"},
    # N1
    "tệ":           {"sentiment_score": -0.6, "ontology_class": "Negative"},
    "kém":          {"sentiment_score": -0.6, "ontology_class": "Negative"},
    "chậm":         {"sentiment_score": -0.6, "ontology_class": "Negative"},
    "thất vọng":    {"sentiment_score": -0.6, "ontology_class": "Negative"},
    "phàn nàn":     {"sentiment_score": -0.6, "ontology_class": "Negative"},
    "khó":          {"sentiment_score": -0.5, "ontology_class": "Negative"},
    "đắt":          {"sentiment_score": -0.5, "ontology_class": "Negative"},
    "bất tiện":     {"sentiment_score": -0.5, "ontology_class": "Negative"},
    "không hài lòng":{"sentiment_score":-0.6, "ontology_class": "Negative"},
    "buồn":         {"sentiment_score": -0.5, "ontology_class": "Negative"},
    "chán":         {"sentiment_score": -0.5, "ontology_class": "Negative"},
    # N2
    "kinh khủng":   {"sentiment_score": -1.0, "ontology_class": "Negative"},
    "tồi tệ":       {"sentiment_score": -1.0, "ontology_class": "Negative"},
    "thảm hại":     {"sentiment_score": -1.0, "ontology_class": "Negative"},
    "ghét":         {"sentiment_score": -1.0, "ontology_class": "Negative"},
    "hỏng":         {"sentiment_score": -1.0, "ontology_class": "Negative"},
    "vô dụng":      {"sentiment_score": -1.0, "ontology_class": "Negative"},
    "lỗi":          {"sentiment_score": -0.8, "ontology_class": "Negative"},
    "gian lận":     {"sentiment_score": -1.0, "ontology_class": "Negative"},
    "lừa đảo":      {"sentiment_score": -1.0, "ontology_class": "Negative"},
    "tồi":          {"sentiment_score": -0.8, "ontology_class": "Negative"},
    "xấu":          {"sentiment_score": -0.7, "ontology_class": "Negative"},
    "khủng khiếp":  {"sentiment_score": -1.0, "ontology_class": "Negative"},
}
INTENSIFIERS_VN = {
    "rất": 1.5, "cực kỳ": 2.0, "vô cùng": 2.0, "hết sức": 1.8,
    "hoàn toàn": 1.8, "thực sự": 1.4, "khá": 1.2, "hơi": 0.7,
    "tương đối": 0.8, "có phần": 0.8, "quá": 1.6,
}
NEGATION_WORDS_VN = {
    "không", "chưa", "chẳng", "không hề", "chẳng hề",
    "không phải", "chưa bao giờ", "không bao giờ",
    "không thể", "không được", "nào có",
}
ONTO_CLASS_IDX = {
    "Positive": 0, "Negative": 1, "Negative_Adverbs": 2, "Prefix": 3,
    "Conjunction": 4, "Coordinating": 5, "Subordinating": 6, "Special_sentence": 7,
    "Noun": 8, "Verb": 9, "Adjective": 10, "Adverb": 11,
    "Emotion": 12, "SentimentAnalysis": 13, "IntensifiersAndNegators": 14, "Unknown": 15,
}


class KGReasoningModuleVN(nn.Module):
    """KG Reasoning Module phiên bản tiếng Việt → h_kg [batch, 64]."""

    def __init__(self, neo4j_uri=NEO4J_URI, neo4j_user=NEO4J_USER,
                 neo4j_password=NEO4J_PASSWORD, kg_dim=KG_DIM):
        super().__init__()
        self.kg_dim         = kg_dim
        self.negation_words = NEGATION_WORDS_VN
        self.driver         = None
        self.embed_proj     = nn.Linear(32, 32)
        self._cache: dict   = {}
        self._logged_error  = False
        self._database      = neo4j_user
        self._connect(neo4j_uri, neo4j_user, neo4j_password)

    def _connect(self, uri, user, password):
        import json as _json
        cache_file = DATA_DIR / "kg_cache.json"
        if cache_file.exists():
            with open(cache_file, "r", encoding="utf-8") as f:
                self._cache = _json.load(f)
            print(f"[KGModule VN] Loaded cache: {len(self._cache)} words")
            return
        print("[KGModule VN] Cache not found -> trying Neo4j...")
        try:
            from neo4j import GraphDatabase
            # Thêm notifications_min_severity="OFF" để tắt warning từ DBMS server
            try:
                self.driver = GraphDatabase.driver(
                    uri, auth=(user, password), notifications_min_severity="OFF"
                )
            except TypeError:
                self.driver = GraphDatabase.driver(uri, auth=(user, password))
            self.driver.verify_connectivity()
            print(f"[KGModule VN] Connected: {uri}")
        except Exception as e:
            print(f"[KGModule VN] Neo4j unavailable: {e}")
            print("[KGModule VN]   Using hardcode VN dict as fallback.")
            self.driver = None

    def _query_word(self, word: str):
        wl = word.lower()
        if wl in FINE_GRAINED_WORD_SCORES_VN: return FINE_GRAINED_WORD_SCORES_VN[wl]
        if wl in STATIC_PHRASE_SCORES_VN:     return STATIC_PHRASE_SCORES_VN[wl]
        if wl in self._cache:                  return self._cache[wl]
        if self.driver is None:                return None
        try:
            with self.driver.session(database=self._database) as s:
                r = s.run("MATCH (w:Word {name:$word}) RETURN w.sentiment_score AS score, w.ontology_class AS cls LIMIT 1", word=wl).single()
                if r and r["score"] is not None:
                    info = {"sentiment_score": float(r["score"]), "ontology_class": r["cls"] or "Unknown"}
                    self._cache[wl] = info
                    return info
        except Exception as e:
            if not self._logged_error:
                print(f"[KGModule VN] Query error: {e}")
                self._logged_error = True
        self._cache[wl] = None
        return None

    def _reason_sentence(self, sentence: str) -> np.ndarray:
        clean_sent = sentence.lower()
        for phrase, (_, _, placeholder) in PHRASE_SENTIMENTS_VN.items():
            if phrase in clean_sent:
                clean_sent = clean_sent.replace(phrase, placeholder)

        has_contrastive    = int(but_scan_vn(clean_sent) != clean_sent)
        after_but_sentence = but_scan_vn(clean_sent)
        before_but_tokens  = clean_up_sentence_vn(clean_sent)
        after_but_tokens   = clean_up_sentence_vn(after_but_sentence)
        tokens = after_but_tokens if has_contrastive else before_but_tokens

        scores, negation_count, negation_countdown = [], 0, 0
        intensifier_count, found_count, onto_classes = 0, 0, []

        i = 0
        while i < len(tokens):
            tok    = tokens[i]
            bigram = " ".join(tokens[i:i+2]) if i < len(tokens)-1 else ""
            if bigram in self.negation_words:
                negation_countdown = 3; negation_count += 1; scores.append(0.0); i += 2; continue
            if tok in self.negation_words:
                negation_countdown = 3; negation_count += 1; scores.append(0.0); i += 1; continue
            intensifier_scale = INTENSIFIERS_VN.get(bigram, INTENSIFIERS_VN.get(tok, 1.0))
            if intensifier_scale != 1.0: intensifier_count += 1
            info = self._query_word(tok)
            if info:
                found_count += 1
                score = info["sentiment_score"] * intensifier_scale
                if negation_countdown > 0:
                    score = -score; negation_countdown = 0
                scores.append(score); onto_classes.append(info["ontology_class"])
            else:
                scores.append(0.0)
            if negation_countdown > 0: negation_countdown -= 1
            i += 1

        n_tokens = max(len(tokens), 1)
        sentiment_scores = [s for s in scores if s != 0.0]
        total_score = float(np.clip(sum(sentiment_scores)/len(sentiment_scores) if sentiment_scores else 0.0, -1.0, 1.0))

        before_scores, after_scores = [0.0], [0.0]
        if has_contrastive:
            for t in clean_up_sentence_vn(clean_sent):
                info = self._query_word(t)
                if info: before_scores.append(info["sentiment_score"])
            for t in clean_up_sentence_vn(after_but_sentence):
                info = self._query_word(t)
                if info: after_scores.append(info["sentiment_score"])

        vec = np.zeros(self.kg_dim, dtype=np.float32)
        vec[0]=total_score; vec[1]=negation_count/n_tokens; vec[2]=intensifier_count/n_tokens
        vec[3]=float(has_contrastive); vec[4]=float(np.clip(np.mean(after_scores),-1,1))
        vec[5]=float(np.clip(np.mean(before_scores),-1,1)); vec[6]=min(negation_count/5.0,1.0)
        vec[7]=found_count/n_tokens
        if onto_classes:
            dom = onto_classes[np.argmax(np.abs(scores[:len(onto_classes)]))]
            vec[8 + ONTO_CLASS_IDX.get(dom, 15)] = 1.0
        if scores:
            sa = np.array(scores)
            f16 = np.array([
                np.mean(sa), np.std(sa), np.max(sa), np.min(sa),
                np.sum(sa>0)/n_tokens, np.sum(sa<0)/n_tokens, np.sum(sa==0)/n_tokens,
                float(np.any(sa>0.5)), float(np.any(sa<-0.5)), len(scores)/50.0,
                float(negation_countdown>0), float(has_contrastive),
                float(intensifier_count>0), float(negation_count>0),
                float(found_count>0), float(found_count/n_tokens>0.5),
            ], dtype=np.float32)
            vec[24:40] = np.clip(f16, -1.0, 1.0)
            n_pos=np.sum(sa>0); n_neg=np.sum(sa<0); n_neu=np.sum(sa==0)
            probs=np.clip(np.array([n_pos,n_neg,n_neu],dtype=np.float32)/max(n_tokens,1),1e-9,1.0)
            entropy=float(-np.sum(probs*np.log(probs+1e-9)))
            f24 = np.array([
                float(np.max(sa)), float(np.min(sa)), float(np.max(np.abs(sa))),
                float(np.sum(np.abs(sa))/n_tokens), float(np.abs(np.mean(sa))), float(np.std(sa)),
                entropy/1.1, float(np.median(sa)), float(n_pos/n_tokens), float(n_neg/n_tokens),
                float(n_neu/n_tokens), float(n_pos/max(n_neg+1,1)), float((n_pos-n_neg)/n_tokens),
                float(negation_count/max(n_pos+1,1)), float(intensifier_count/n_tokens),
                float(found_count/n_tokens), float(has_contrastive),
                float(np.clip(np.mean(before_scores),-1,1)), float(np.clip(np.mean(after_scores),-1,1)),
                float(np.clip(np.mean(after_scores)-np.mean(before_scores),-2,2)/2.0),
                float(np.abs(np.mean(after_scores)-np.mean(before_scores))),
                float(np.mean(after_scores)>np.mean(before_scores)),
                float(np.mean(after_scores)<np.mean(before_scores)), float(negation_count>1),
            ], dtype=np.float32)
            vec[40:64] = np.clip(f24, -1.0, 1.0)
        return vec

    def forward(self, sentences: list) -> torch.Tensor:
        vecs = [self._reason_sentence(s) for s in sentences]
        return torch.tensor(np.stack(vecs), dtype=torch.float32).to(next(self.parameters()).device)

    def __del__(self):
        if self.driver: self.driver.close()


if __name__ == "__main__":
    kg = KGReasoningModuleVN().to(DEVICE)
    for w in ["tốt", "tệ", "không", "tuyệt vời", "kinh khủng"]:
        print(f"  '{w}': {kg._query_word(w)}")
    h = kg(["sản phẩm không tệ chút nào", "hoàn toàn kinh khủng", "dịch vụ tốt nhưng giao hàng chậm"])
    print(f"h_kg shape: {h.shape} | non-zero: {(h!=0).sum().item()}/{h.numel()}")


[KGModule VN] Cache not found -> trying Neo4j...
[KGModule VN] Neo4j unavailable: Failed to DNS resolve address 9f1d22e9.databases.neo4j.io:7687: [Errno -2] Name or service not known
[KGModule VN]   Using hardcode VN dict as fallback.
  'tốt': {'sentiment_score': 0.6, 'ontology_class': 'Positive'}
  'tệ': {'sentiment_score': -0.6, 'ontology_class': 'Negative'}
  'không': None
  'tuyệt vời': {'sentiment_score': 1.0, 'ontology_class': 'Positive'}
  'kinh khủng': {'sentiment_score': -1.0, 'ontology_class': 'Negative'}
h_kg shape: torch.Size([3, 64]) | non-zero: 62/192


## 6. models/phobert_encoder.py

In [6]:
"""
models/phobert_encoder.py — PhoBERT Encoder.
Thay bert-base-uncased bằng vinai/phobert-base.
PhoBERT dùng BPE tokenizer (sentencepiece), giữ cùng interface.
"""
from __future__ import annotations
import torch, torch.nn as nn
from transformers import AutoTokenizer, AutoModel


class PhoBERTEncoder(nn.Module):
    """Encode câu tiếng Việt bằng PhoBERT. Output h_bert [batch, 768]."""

    def __init__(self, model_name=None, max_length=None, freeze_layers=None, pooling=None):
        super().__init__()
        model_name    = model_name    or BERT_MODEL_NAME
        max_length    = max_length    or BERT_MAX_LENGTH
        freeze_layers = freeze_layers if freeze_layers is not None else BERT_FREEZE_LAYERS
        pooling       = pooling       or BERT_POOLING

        self.tokenizer  = AutoTokenizer.from_pretrained(model_name)
        self.bert       = AutoModel.from_pretrained(model_name)
        self.max_length = max_length
        self.pooling    = pooling

        self._freeze_layers(freeze_layers)
        print(f"[PhoBERTEncoder] Loaded {model_name} | Freeze {freeze_layers} layers | "
              f"Pooling: {pooling} | Trainable: {self._count_trainable():,}")

    def _freeze_layers(self, n):
        for p in self.bert.embeddings.parameters(): p.requires_grad = False
        if hasattr(self.bert, "encoder") and hasattr(self.bert.encoder, "layer"):
            layers = self.bert.encoder.layer
        elif hasattr(self.bert, "transformer") and hasattr(self.bert.transformer, "layer"):
            layers = self.bert.transformer.layer
        else:
            print("[PhoBERTEncoder] Warning: không tìm thấy layers để freeze"); return
        for i, layer in enumerate(layers):
            if i < n:
                for p in layer.parameters(): p.requires_grad = False

    def _count_trainable(self): return sum(p.numel() for p in self.parameters() if p.requires_grad)

    def forward(self, sentences: list) -> torch.Tensor:
        encoded = self.tokenizer(sentences, padding=True, truncation=True,
                                 max_length=self.max_length, return_tensors="pt")
        dev = next(self.parameters()).device
        input_ids      = encoded["input_ids"].to(dev)
        attention_mask = encoded["attention_mask"].to(dev)
        outputs     = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden = outputs.last_hidden_state
        if self.pooling == "mean_cls":
            cls_emb       = last_hidden[:, 0, :]
            mask_expanded = attention_mask.unsqueeze(-1).float()
            mean_emb      = (last_hidden * mask_expanded).sum(1) / mask_expanded.sum(1).clamp(min=1e-9)
            return 0.5 * cls_emb + 0.5 * mean_emb
        return last_hidden[:, 0, :]


if __name__ == "__main__":
    enc = PhoBERTEncoder().to(DEVICE)
    h = enc(["sản phẩm tuyệt vời", "hoàn toàn kinh khủng"])
    print(f"h_bert shape: {h.shape}")


config.json:   0%|          | 0.00/557 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/543M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: vinai/phobert-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
lm_head.decoder.weight          | UNEXPECTED |  | 
lm_head.decoder.bias            | UNEXPECTED |  | 
lm_head.dense.weight            | UNEXPECTED |  | 
lm_head.layer_norm.weight       | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 
lm_head.layer_norm.bias         | UNEXPECTED |  | 
lm_head.bias                    | UNEXPECTED |  | 
lm_head.dense.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[PhoBERTEncoder] Loaded vinai/phobert-base | Freeze 2 layers | Pooling: mean_cls | Trainable: 71,469,312


model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

h_bert shape: torch.Size([2, 768])


## 7. models/fusion.py + ota_layer.py

In [7]:
"""
models/fusion.py + ota_layer.py — Giữ nguyên từ bản EN (không phụ thuộc ngôn ngữ).
"""
from __future__ import annotations
import torch, torch.nn as nn


class MLPFusion(nn.Module):
    def __init__(self, bert_dim=BERT_HIDDEN_DIM, kg_dim=KG_DIM, hidden_dim=MLP_HIDDEN_DIM, dropout=MLP_DROPOUT):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(bert_dim+kg_dim, 512), nn.LayerNorm(512), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(512, hidden_dim), nn.ReLU(), nn.Dropout(dropout),
        )
        self.output_dim = hidden_dim
    def forward(self, h_bert, h_kg): return self.mlp(torch.cat([h_bert, h_kg], dim=-1))


class GatedFusion(nn.Module):
    def __init__(self, bert_dim=BERT_HIDDEN_DIM, kg_dim=KG_DIM, kg_mid=256, dropout=MLP_DROPOUT):
        super().__init__()
        self.kg_proj = nn.Sequential(
            nn.Linear(kg_dim, kg_mid), nn.ReLU(), nn.Dropout(dropout*0.5), nn.Linear(kg_mid, bert_dim),
        )
        self.kg_norm    = nn.LayerNorm(bert_dim)
        self.kg_dropout = nn.Dropout(dropout*0.5)
        self.gate       = nn.Linear(bert_dim*2, bert_dim)
        self.dropout    = nn.Dropout(dropout)
        self.out_norm   = nn.LayerNorm(bert_dim)
        nn.init.zeros_(self.gate.bias); nn.init.xavier_uniform_(self.gate.weight, gain=0.1)
        self.output_dim = bert_dim
    def forward(self, h_bert, h_kg):
        h_kg_proj = self.kg_norm(self.kg_proj(h_kg))
        h_kg_drop = self.kg_dropout(h_kg_proj)
        g = torch.sigmoid(self.gate(self.dropout(torch.cat([h_bert, h_kg_drop], dim=-1))))
        return self.out_norm(g * h_bert + (1.0 - g) * h_kg_proj)


def get_fusion(fusion_type: str, **kwargs) -> nn.Module:
    if fusion_type == "mlp":   return MLPFusion(**kwargs)
    if fusion_type == "gated": return GatedFusion(**kwargs)
    raise ValueError(f"fusion_type: {fusion_type}")


class OTALayer(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, kg_dim=KG_DIM, alpha=OTA_ALPHA, learnable_alpha=OTA_LEARNABLE_ALPHA):
        super().__init__()
        self.adjustment = nn.Linear(kg_dim, num_classes)
        if learnable_alpha: self.alpha = nn.Parameter(torch.tensor(alpha))
        else:               self.register_buffer("alpha", torch.tensor(alpha))
        self.layer_norm    = nn.LayerNorm(num_classes)
        self.residual_gate = nn.Parameter(torch.tensor(-2.0))
        print(f"[OTALayer] alpha={'learnable' if learnable_alpha else alpha} | kg_dim={kg_dim} -> num_classes={num_classes}")
    def forward(self, logits, h_kg):
        adj  = self.layer_norm(self.adjustment(h_kg))
        gate = torch.sigmoid(self.residual_gate)
        return logits + gate * self.alpha * adj


print("[Fusion + OTALayer] Loaded OK")


[Fusion + OTALayer] Loaded OK


## 8. models/hybrid_model_vn.py

In [8]:
"""
models/hybrid_model_vn.py — HybridSentimentModelVN (PhoBERT + KG_VN + Fusion + OTA).
Pipeline giống hệt bản EN, chỉ thay BERTEncoder→PhoBERTEncoder, KGModule→KGModuleVN.
"""
from __future__ import annotations
import torch, torch.nn as nn, torch.nn.functional as F


class HybridSentimentModelVN(nn.Module):
    """
    Pipeline:
        sentences
            ├── PhoBERTEncoder      → h_bert [batch, 768]
            └── KGReasoningModuleVN → h_kg  [batch, 64]
                    ↓
               Fusion (mlp|gated) → h_fusion [batch, hidden_dim]
                    ↓
               Linear classifier  → logits [batch, 4]
                    ↓
               OTALayer           → adjusted_logits [batch, 4]
                    ↓
               Softmax            → probs [batch, 4]
    """

    def __init__(self, fusion_type=FUSION_TYPE, num_classes=NUM_CLASSES, kg_dim=KG_DIM,
                 neo4j_uri=NEO4J_URI, neo4j_user=NEO4J_USER, neo4j_password=NEO4J_PASSWORD):
        super().__init__()
        self.fusion_type   = fusion_type
        self.bert_encoder  = PhoBERTEncoder()
        self.kg_module     = KGReasoningModuleVN(neo4j_uri, neo4j_user, neo4j_password, kg_dim)
        self.fusion        = get_fusion(fusion_type)
        hidden_dim         = self.fusion.output_dim
        self.class_dropout = nn.Dropout(0.2)
        self.classifier    = nn.Linear(hidden_dim, num_classes)
        self.ota_layer     = OTALayer(num_classes=num_classes, kg_dim=kg_dim)

        total     = sum(p.numel() for p in self.parameters())
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"\n[HybridModelVN] Fusion: {fusion_type} | Hidden: {hidden_dim} | "
              f"Total: {total:,} | Trainable: {trainable:,}")

    def forward(self, sentences: list):
        h_bert          = self.bert_encoder(sentences)
        h_kg            = self.kg_module(sentences).to(h_bert.device)
        h_fusion        = self.fusion(h_bert, h_kg)
        logits          = self.classifier(self.class_dropout(h_fusion))
        adjusted_logits = self.ota_layer(logits, h_kg)
        probs           = F.softmax(adjusted_logits, dim=-1)
        return probs, adjusted_logits

    def predict(self, sentence: str) -> dict:
        self.eval()
        with torch.no_grad():
            probs, _ = self.forward([sentence])
        probs_np = probs[0].cpu().numpy()
        pred_id  = probs_np.argmax()
        return {
            "label":      ID2LABEL[int(pred_id)],
            "confidence": float(probs_np[pred_id]),
            "all_probs":  {ID2LABEL[i]: float(p) for i, p in enumerate(probs_np)},
        }


if __name__ == "__main__":
    model = HybridSentimentModelVN().to(DEVICE)
    model.eval()
    with torch.no_grad():
        probs, _ = model(["sản phẩm không tệ chút nào", "hoàn toàn kinh khủng"])
    for s, p in zip(["câu 1","câu 2"], probs):
        print(f"  '{s}' -> {ID2LABEL[p.argmax().item()]} ({p.max().item():.3f})")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: vinai/phobert-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
lm_head.decoder.weight          | UNEXPECTED |  | 
lm_head.decoder.bias            | UNEXPECTED |  | 
lm_head.dense.weight            | UNEXPECTED |  | 
lm_head.layer_norm.weight       | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 
lm_head.layer_norm.bias         | UNEXPECTED |  | 
lm_head.bias                    | UNEXPECTED |  | 
lm_head.dense.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[PhoBERTEncoder] Loaded vinai/phobert-base | Freeze 2 layers | Pooling: mean_cls | Trainable: 71,469,312
[KGModule VN] Cache not found -> trying Neo4j...
[KGModule VN] Neo4j unavailable: Failed to DNS resolve address 9f1d22e9.databases.neo4j.io:7687: [Errno -2] Name or service not known
[KGModule VN]   Using hardcode VN dict as fallback.
[OTALayer] alpha=learnable | kg_dim=64 -> num_classes=4

[HybridModelVN] Fusion: mlp | Hidden: 384 | Total: 135,625,650 | Trainable: 72,096,690
  'câu 1' -> N1 (0.303)
  'câu 2' -> N1 (0.296)


## 9. utils/metrics.py

In [9]:
"""
utils/metrics.py — Giữ nguyên từ bản EN.
"""
from __future__ import annotations
import torch, numpy as np
from sklearn.metrics import classification_report, f1_score, accuracy_score


def evaluate_model(model, dataloader, device) -> dict:
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in dataloader:
            sentences = batch["sentence"]
            labels    = batch["label"].to(device)
            _, logits = model(sentences)
            preds = logits.argmax(dim=-1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)
    acc        = accuracy_score(all_labels, all_preds)
    f1_macro   = f1_score(all_labels, all_preds, average="macro",    zero_division=0)
    f1_w       = f1_score(all_labels, all_preds, average="weighted", zero_division=0)
    report     = classification_report(
        all_labels, all_preds,
        target_names=[ID2LABEL[i] for i in range(NUM_CLASSES)], zero_division=0,
    )
    print(f"\n{'='*50}")
    print(f"Accuracy   : {acc:.4f}")
    print(f"F1 Macro   : {f1_macro:.4f}")
    print(f"F1 Weighted: {f1_w:.4f}")
    print(f"\n{report}")
    print('='*50)
    return {"accuracy": acc, "f1_macro": f1_macro, "f1_weighted": f1_w, "report": report}


print("[Metrics] Loaded OK")


[Metrics] Loaded OK


## 10. train_vn.py

In [10]:
"""
train_vn.py — Training loop cho HybridSentimentModelVN.
Schema: N1/N2/P1/P2. Data: TranfomerVN.json.
"""
from __future__ import annotations
import torch, torch.nn as nn, random, numpy as np, math
from torch.utils.data import WeightedRandomSampler


def set_seed(seed=RANDOM_SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)


def get_class_weights_vn(dataset):
    counts = [0]*NUM_CLASSES
    for s in dataset.samples: counts[s["label"]] += 1
    max_c   = max(counts)
    weights = [math.sqrt(max_c/c) if c > 0 else 1.0 for c in counts]
    mean_w  = sum(weights)/len(weights)
    weights = [w/mean_w for w in weights]
    print(f"[Class Weights VN] counts={counts}")
    print(f"[Class Weights VN] weights={[f'{w:.3f}' for w in weights]}")
    return torch.tensor(weights, dtype=torch.float32)


def get_weighted_dataloader_vn(dataset, batch_size, train_indices):
    class_counts = [0]*NUM_CLASSES
    for idx in train_indices: class_counts[dataset[idx]["label"].item()] += 1
    cw = [1.0/c if c > 0 else 0.0 for c in class_counts]
    sw = torch.tensor([cw[dataset[idx]["label"].item()] for idx in train_indices], dtype=torch.float32)
    sampler = WeightedRandomSampler(sw, len(sw), replacement=True)
    return torch.utils.data.DataLoader(
        torch.utils.data.Subset(dataset, train_indices),
        batch_size=batch_size, sampler=sampler, num_workers=0, pin_memory=True,
    )


def get_warmup_scheduler(optimizer, warmup_steps, total_steps):
    def lr_lambda(step):
        if step < warmup_steps: return float(step)/float(max(1,warmup_steps))
        progress = float(step-warmup_steps)/float(max(1,total_steps-warmup_steps))
        return max(0.0, 0.5*(1.0+np.cos(np.pi*progress)))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


def train_vn(fusion_type: str = FUSION_TYPE) -> None:
    set_seed()
    print(f"\n{'='*60}")
    print(f"Training HybridSentimentModelVN (PhoBERT) | fusion={fusion_type}")
    print(f"Device: {DEVICE}  |  Schema: N1/N2/P1/P2  |  Data: {INTENT_FILE.name}")
    print(f"{'='*60}\n")

    dataset = SentimentDatasetVN(INTENT_FILE)
    n_train = int(len(dataset)*TRAIN_VAL_SPLIT)
    n_val   = len(dataset)-n_train
    gen     = torch.Generator().manual_seed(RANDOM_SEED)
    train_set, val_set = torch.utils.data.random_split(dataset, [n_train, n_val], generator=gen)

    train_loader = get_weighted_dataloader_vn(dataset, BATCH_SIZE, train_set.indices)
    val_loader   = torch.utils.data.DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
    print(f"[DataLoader VN] Train: {n_train} | Val: {n_val}")

    model = HybridSentimentModelVN(fusion_type=fusion_type).to(DEVICE)

    cw        = get_class_weights_vn(dataset).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=cw, label_smoothing=LABEL_SMOOTHING)
    print(f"[Loss] CrossEntropyLoss(label_smoothing={LABEL_SMOOTHING}, class_weights=True)")

    bert_params  = list(model.bert_encoder.parameters())
    other_params = (list(model.kg_module.parameters()) + list(model.fusion.parameters()) +
                    list(model.classifier.parameters()) + list(model.ota_layer.parameters()))
    optimizer = torch.optim.AdamW([
        {"params": bert_params,  "lr": LR_BERT},
        {"params": other_params, "lr": LR_REST},
    ], weight_decay=WEIGHT_DECAY)

    total_steps = NUM_EPOCHS*len(train_loader)
    scheduler   = get_warmup_scheduler(optimizer, int(total_steps*0.1), total_steps)

    best_f1 = 0.0; no_improve = 0
    ckpt = CHECKPOINT_DIR / f"model_vn_{fusion_type}.pt"

    for epoch in range(1, NUM_EPOCHS+1):
        model.train(); total_loss = correct = total = 0
        for batch_idx, batch in enumerate(train_loader):
            sentences = batch["sentence"]; labels = batch["label"].to(DEVICE)
            optimizer.zero_grad()
            _, logits = model(sentences)
            loss = criterion(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step(); scheduler.step()
            total_loss += loss.item()
            preds   = logits.argmax(dim=-1)
            correct += (preds==labels).sum().item(); total += len(labels)
            if (batch_idx+1) % 50 == 0:
                print(f"  Epoch {epoch}/{NUM_EPOCHS} | Step {batch_idx+1}/{len(train_loader)} | "
                      f"Loss: {loss.item():.4f} | LR: {scheduler.get_last_lr()[0]:.2e}")

        print(f"\nEpoch {epoch}/{NUM_EPOCHS} | Train Loss: {total_loss/len(train_loader):.4f} | Train Acc: {correct/total:.4f}")
        metrics = evaluate_model(model, val_loader, DEVICE)
        val_f1  = metrics["f1_macro"]
        if val_f1 > best_f1:
            best_f1 = val_f1; no_improve = 0
            torch.save(model.state_dict(), ckpt)
            print(f"  Saved best (F1={best_f1:.4f}) -> {ckpt}")
        else:
            no_improve += 1
            print(f"  No improvement ({no_improve}/{EARLY_STOPPING_PATIENCE})")
            if no_improve >= EARLY_STOPPING_PATIENCE:
                print(f"\nEarly stopping tại epoch {epoch}"); break

    print(f"\n{'='*60}")
    print(f"Training xong! Best F1 Macro: {best_f1:.4f}")
    print(f"Checkpoint: {ckpt}")
    print(f"{'='*60}")


if __name__ == "__main__":
    train_vn(fusion_type="mlp")



Training HybridSentimentModelVN (PhoBERT) | fusion=mlp
Device: cuda  |  Schema: N1/N2/P1/P2  |  Data: TranfomerVN.json

  [Domain VI] Loaded 684 unique Vietnamese domain samples.
  [Domain VI] Added 3420 domain samples (x5).

[Dataset VN] Loaded 10052 samples from TranfomerVN.json
  (original: 6934, augmented: 3118)
  Label distribution:
  N1: 1828 samples  ####################################
  N2: 1828 samples  ####################################
  P1: 1488 samples  #############################
  P2: 1488 samples  #############################
[DataLoader VN] Train: 8041 | Val: 2011


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: vinai/phobert-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
lm_head.decoder.weight          | UNEXPECTED |  | 
lm_head.decoder.bias            | UNEXPECTED |  | 
lm_head.dense.weight            | UNEXPECTED |  | 
lm_head.layer_norm.weight       | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 
lm_head.layer_norm.bias         | UNEXPECTED |  | 
lm_head.bias                    | UNEXPECTED |  | 
lm_head.dense.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[PhoBERTEncoder] Loaded vinai/phobert-base | Freeze 2 layers | Pooling: mean_cls | Trainable: 71,469,312
[KGModule VN] Cache not found -> trying Neo4j...
[KGModule VN] Neo4j unavailable: Failed to DNS resolve address 9f1d22e9.databases.neo4j.io:7687: [Errno -2] Name or service not known
[KGModule VN]   Using hardcode VN dict as fallback.
[OTALayer] alpha=learnable | kg_dim=64 -> num_classes=4

[HybridModelVN] Fusion: mlp | Hidden: 384 | Total: 135,625,650 | Trainable: 72,096,690
[Class Weights VN] counts=[3013, 3013, 2018, 2008]
[Class Weights VN] weights=['0.900', '0.900', '1.099', '1.102']
[Loss] CrossEntropyLoss(label_smoothing=0.1, class_weights=True)
  Epoch 1/30 | Step 50/503 | Loss: 1.3465 | LR: 6.63e-07
  Epoch 1/30 | Step 100/503 | Loss: 1.4659 | LR: 1.33e-06
  Epoch 1/30 | Step 150/503 | Loss: 1.4841 | LR: 1.99e-06
  Epoch 1/30 | Step 200/503 | Loss: 1.4269 | LR: 2.65e-06
  Epoch 1/30 | Step 250/503 | Loss: 1.3231 | LR: 3.31e-06
  Epoch 1/30 | Step 300/503 | Loss: 1.3778 | LR

## 11. evaluate_vn.py

In [11]:
"""
evaluate_vn.py — Đánh giá HybridSentimentModelVN trên tập val và tập test tiếng Việt (Testing_VI.txt / ReadAcc_VI.txt).
"""
from __future__ import annotations
import torch, numpy as np
from pathlib import Path
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score


def evaluate_vn(fusion_type: str = FUSION_TYPE) -> None:
    print(f"\n{'='*56}")
    print(f"  QoE - HybridSentimentModelVN | Schema N1/N2/P1/P2")
    print(f"  Input: {INTENT_FILE.name} | Device: {DEVICE} | Fusion: {fusion_type}")
    print(f"{'='*56}\n")

    print("[Data] Loading validation split...")
    dataset   = SentimentDatasetVN(INTENT_FILE)
    n_train   = int(len(dataset)*TRAIN_VAL_SPLIT)
    n_val     = len(dataset)-n_train
    gen       = torch.Generator().manual_seed(RANDOM_SEED)
    _, val_set = torch.utils.data.random_split(dataset, [n_train, n_val], generator=gen)
    val_loader = torch.utils.data.DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    print(f"  Val split: {n_val} samples")

    ckpt  = CHECKPOINT_DIR / f"model_vn_{fusion_type}.pt"
    model = HybridSentimentModelVN(fusion_type=fusion_type).to(DEVICE)
    if ckpt.exists():
        model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
        print(f"  Loaded: {ckpt.name}")
    else:
        print(f"  [!] Checkpoint không tồn tại: {ckpt}")
        print("  Hãy chạy train_vn() trước."); return

    print(f"\n  [Val Split] {n_val} samples...")
    evaluate_model(model, val_loader, DEVICE)

    # ── Test trên dữ liệu tiếng Việt thực tế: Testing_VI.txt & ReadAcc_VI.txt ───
    print(f"\n[Data] Loading Vietnamese test files...")
    test_files = [TESTING_ACC_FILE, TESTING_ACC2_FILE]
    domain = []
    for fn in test_files:
        if fn.exists():
            samp = _load_domain_file(fn)
            print(f"  {fn.name:30s}: {len(samp)} Vietnamese sentences")
            domain.extend(samp)

    if domain:
        model.eval(); preds_all, labels_all = [], []
        with torch.no_grad():
            for i in range(0, len(domain), BATCH_SIZE):
                batch = domain[i:i+BATCH_SIZE]
                _, logits = model([s["sentence"] for s in batch])
                preds_all.extend(logits.argmax(dim=-1).cpu().numpy())
                labels_all.extend([s["label"] for s in batch])
        preds_all  = np.array(preds_all); labels_all = np.array(labels_all)
        acc = accuracy_score(labels_all, preds_all)
        f1  = f1_score(labels_all, preds_all, average="macro", zero_division=0)
        rep = classification_report(labels_all, preds_all, target_names=[ID2LABEL[i] for i in range(NUM_CLASSES)], zero_division=0)
        print(f"\n{'='*56}")
        print(f"  [Test Real_VI] {len(domain)} Vietnamese samples")
        print(f"  Accuracy   : {acc:.4f} ({acc*100:.2f}%)")
        print(f"  F1 Macro   : {f1:.4f}")
        print(f"\n{rep}")
        cm = confusion_matrix(labels_all, preds_all)
        names = [ID2LABEL[i] for i in range(NUM_CLASSES)]
        print("  Confusion Matrix (rows=True, cols=Pred):")
        print("          " + "  ".join(f"{n:>4}" for n in names))
        for i, row in enumerate(cm):
            print(f"    {names[i]:>4}  " + "  ".join(f"{v:>4}" for v in row))
        print('='*56)


if __name__ == "__main__":
    evaluate_vn(fusion_type="mlp")



  QoE - HybridSentimentModelVN | Schema N1/N2/P1/P2
  Input: TranfomerVN.json | Device: cuda | Fusion: mlp

[Data] Loading validation split...
  [Domain VI] Loaded 684 unique Vietnamese domain samples.
  [Domain VI] Added 3420 domain samples (x5).

[Dataset VN] Loaded 10052 samples from TranfomerVN.json
  (original: 6934, augmented: 3118)
  Label distribution:
  N1: 1828 samples  ####################################
  N2: 1828 samples  ####################################
  P1: 1488 samples  #############################
  P2: 1488 samples  #############################
  Val split: 2011 samples


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: vinai/phobert-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
lm_head.decoder.weight          | UNEXPECTED |  | 
lm_head.decoder.bias            | UNEXPECTED |  | 
lm_head.dense.weight            | UNEXPECTED |  | 
lm_head.layer_norm.weight       | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 
lm_head.layer_norm.bias         | UNEXPECTED |  | 
lm_head.bias                    | UNEXPECTED |  | 
lm_head.dense.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[PhoBERTEncoder] Loaded vinai/phobert-base | Freeze 2 layers | Pooling: mean_cls | Trainable: 71,469,312
[KGModule VN] Cache not found -> trying Neo4j...
[KGModule VN] Neo4j unavailable: Failed to DNS resolve address 9f1d22e9.databases.neo4j.io:7687: [Errno -2] Name or service not known
[KGModule VN]   Using hardcode VN dict as fallback.
[OTALayer] alpha=learnable | kg_dim=64 -> num_classes=4

[HybridModelVN] Fusion: mlp | Hidden: 384 | Total: 135,625,650 | Trainable: 72,096,690
  Loaded: model_vn_mlp.pt

  [Val Split] 2011 samples...

Accuracy   : 0.8463
F1 Macro   : 0.8497
F1 Weighted: 0.8462

              precision    recall  f1-score   support

          N1       0.84      0.81      0.83       600
          N2       0.83      0.85      0.84       603
          P1       0.86      0.87      0.86       413
          P2       0.87      0.87      0.87       395

    accuracy                           0.85      2011
   macro avg       0.85      0.85      0.85      2011
weighted avg     

## 12. Chạy Pipeline

Chạy các cell bên dưới theo thứ tự.

In [12]:
# ── Bước 1: Test load dataset tiếng Việt ──────────────────────────────────────
print("[Test] Loading TranfomerVN.json...")
ds_vn = SentimentDatasetVN(INTENT_FILE)
s = ds_vn[0]
print(f"\nSample đầu tiên: '{s['sentence']}' -> {ID2LABEL[s['label'].item()]}")
print(f"Tổng samples: {len(ds_vn)}")


[Test] Loading TranfomerVN.json...
  [Domain VI] Loaded 684 unique Vietnamese domain samples.
  [Domain VI] Added 3420 domain samples (x5).

[Dataset VN] Loaded 10052 samples from TranfomerVN.json
  (original: 6934, augmented: 3118)
  Label distribution:
  N1: 1828 samples  ####################################
  N2: 1828 samples  ####################################
  P1: 1488 samples  #############################
  P2: 1488 samples  #############################

Sample đầu tiên: 'tồn tại trong thời gian ngắn' -> N1
Tổng samples: 10052


In [13]:
# ── Bước 2: Training (bỏ comment để chạy) ────────────────────────────────────
train_vn(fusion_type="mlp")
# train_vn(fusion_type="gated")
print("Bỏ comment dòng train_vn() để bắt đầu training.")



Training HybridSentimentModelVN (PhoBERT) | fusion=mlp
Device: cuda  |  Schema: N1/N2/P1/P2  |  Data: TranfomerVN.json

  [Domain VI] Loaded 684 unique Vietnamese domain samples.
  [Domain VI] Added 3420 domain samples (x5).

[Dataset VN] Loaded 10052 samples from TranfomerVN.json
  (original: 6934, augmented: 3118)
  Label distribution:
  N1: 1828 samples  ####################################
  N2: 1828 samples  ####################################
  P1: 1488 samples  #############################
  P2: 1488 samples  #############################
[DataLoader VN] Train: 8041 | Val: 2011


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: vinai/phobert-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
lm_head.decoder.weight          | UNEXPECTED |  | 
lm_head.decoder.bias            | UNEXPECTED |  | 
lm_head.dense.weight            | UNEXPECTED |  | 
lm_head.layer_norm.weight       | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 
lm_head.layer_norm.bias         | UNEXPECTED |  | 
lm_head.bias                    | UNEXPECTED |  | 
lm_head.dense.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[PhoBERTEncoder] Loaded vinai/phobert-base | Freeze 2 layers | Pooling: mean_cls | Trainable: 71,469,312
[KGModule VN] Cache not found -> trying Neo4j...
[KGModule VN] Neo4j unavailable: Failed to DNS resolve address 9f1d22e9.databases.neo4j.io:7687: [Errno -2] Name or service not known
[KGModule VN]   Using hardcode VN dict as fallback.
[OTALayer] alpha=learnable | kg_dim=64 -> num_classes=4

[HybridModelVN] Fusion: mlp | Hidden: 384 | Total: 135,625,650 | Trainable: 72,096,690
[Class Weights VN] counts=[3013, 3013, 2018, 2008]
[Class Weights VN] weights=['0.900', '0.900', '1.099', '1.102']
[Loss] CrossEntropyLoss(label_smoothing=0.1, class_weights=True)
  Epoch 1/30 | Step 50/503 | Loss: 1.3465 | LR: 6.63e-07
  Epoch 1/30 | Step 100/503 | Loss: 1.4659 | LR: 1.33e-06
  Epoch 1/30 | Step 150/503 | Loss: 1.4841 | LR: 1.99e-06
  Epoch 1/30 | Step 200/503 | Loss: 1.4269 | LR: 2.65e-06
  Epoch 1/30 | Step 250/503 | Loss: 1.3231 | LR: 3.31e-06
  Epoch 1/30 | Step 300/503 | Loss: 1.3778 | LR

In [14]:
# ── Bước 3: Đánh giá trên tập test tiếng Việt (bỏ comment sau khi train) ──────
evaluate_vn(fusion_type="mlp")
print("Bỏ comment dòng evaluate_vn() để chạy đánh giá.")



  QoE - HybridSentimentModelVN | Schema N1/N2/P1/P2
  Input: TranfomerVN.json | Device: cuda | Fusion: mlp

[Data] Loading validation split...
  [Domain VI] Loaded 684 unique Vietnamese domain samples.
  [Domain VI] Added 3420 domain samples (x5).

[Dataset VN] Loaded 10052 samples from TranfomerVN.json
  (original: 6934, augmented: 3118)
  Label distribution:
  N1: 1828 samples  ####################################
  N2: 1828 samples  ####################################
  P1: 1488 samples  #############################
  P2: 1488 samples  #############################
  Val split: 2011 samples


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: vinai/phobert-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
lm_head.decoder.weight          | UNEXPECTED |  | 
lm_head.decoder.bias            | UNEXPECTED |  | 
lm_head.dense.weight            | UNEXPECTED |  | 
lm_head.layer_norm.weight       | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 
lm_head.layer_norm.bias         | UNEXPECTED |  | 
lm_head.bias                    | UNEXPECTED |  | 
lm_head.dense.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[PhoBERTEncoder] Loaded vinai/phobert-base | Freeze 2 layers | Pooling: mean_cls | Trainable: 71,469,312
[KGModule VN] Cache not found -> trying Neo4j...
[KGModule VN] Neo4j unavailable: Failed to DNS resolve address 9f1d22e9.databases.neo4j.io:7687: [Errno -2] Name or service not known
[KGModule VN]   Using hardcode VN dict as fallback.
[OTALayer] alpha=learnable | kg_dim=64 -> num_classes=4

[HybridModelVN] Fusion: mlp | Hidden: 384 | Total: 135,625,650 | Trainable: 72,096,690
  Loaded: model_vn_mlp.pt

  [Val Split] 2011 samples...

Accuracy   : 0.8463
F1 Macro   : 0.8497
F1 Weighted: 0.8462

              precision    recall  f1-score   support

          N1       0.84      0.81      0.83       600
          N2       0.83      0.85      0.84       603
          P1       0.86      0.87      0.86       413
          P2       0.87      0.87      0.87       395

    accuracy                           0.85      2011
   macro avg       0.85      0.85      0.85      2011
weighted avg     

## 13. Inference Demo

In [15]:
# ── Demo Inference tiếng Việt ──────────────────────────────────────────────────

def demo_predict_vn(sentences: list, fusion_type: str = "mlp") -> None:
    """Demo phân tích cảm xúc câu tiếng Việt."""
    ckpt = CHECKPOINT_DIR / f"model_vn_{fusion_type}.pt"
    if not ckpt.exists():
        print(f"[!] Chưa có checkpoint. Chạy train_vn() trước."); return

    model = HybridSentimentModelVN(fusion_type=fusion_type).to(DEVICE)
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))

    label_vn = {
        "N1": "😐 Tiêu cực nhẹ",  "N2": "😠 Tiêu cực nặng",
        "P1": "🙂 Tích cực nhẹ",   "P2": "😄 Tích cực mạnh",
    }
    print(f"\n{'='*60}\n  Demo Phân Tích Cảm Xúc Tiếng Việt (PhoBERT + KG)\n{'='*60}")
    for sentence in sentences:
        result = model.predict(sentence)
        label  = result["label"]; conf = result["confidence"]; ap = result["all_probs"]
        print(f"\nCâu   : {sentence}")
        print(f"Nhãn  : {label_vn.get(label, label)} (confidence: {conf:.2%})")
        print(f"Probs : " + " | ".join(f"{k}={v:.2%}" for k, v in ap.items()))


# Bỏ comment để thử:
demo_predict_vn([
    "Sản phẩm tuyệt vời, giao hàng nhanh, rất hài lòng!",
    "Chất lượng kém, vỡ sau 1 ngày, không mua lại nữa.",
    "Tạm ổn, không có gì đặc biệt nhưng giá cả hợp lý.",
    "Hoàn toàn kinh khủng! Lừa đảo! Tránh xa!",
    "Khá tốt, đáng tiền mua.",
    ])

print("[Demo] Cell sẵn sàng. Bỏ comment demo_predict_vn() và chạy sau khi train xong.")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: vinai/phobert-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
lm_head.decoder.weight          | UNEXPECTED |  | 
lm_head.decoder.bias            | UNEXPECTED |  | 
lm_head.dense.weight            | UNEXPECTED |  | 
lm_head.layer_norm.weight       | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 
lm_head.layer_norm.bias         | UNEXPECTED |  | 
lm_head.bias                    | UNEXPECTED |  | 
lm_head.dense.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[PhoBERTEncoder] Loaded vinai/phobert-base | Freeze 2 layers | Pooling: mean_cls | Trainable: 71,469,312
[KGModule VN] Cache not found -> trying Neo4j...
[KGModule VN] Neo4j unavailable: Failed to DNS resolve address 9f1d22e9.databases.neo4j.io:7687: [Errno -2] Name or service not known
[KGModule VN]   Using hardcode VN dict as fallback.
[OTALayer] alpha=learnable | kg_dim=64 -> num_classes=4

[HybridModelVN] Fusion: mlp | Hidden: 384 | Total: 135,625,650 | Trainable: 72,096,690

  Demo Phân Tích Cảm Xúc Tiếng Việt (PhoBERT + KG)

Câu   : Sản phẩm tuyệt vời, giao hàng nhanh, rất hài lòng!
Nhãn  : 😄 Tích cực mạnh (confidence: 92.52%)
Probs : N1=1.91% | N2=3.13% | P1=2.44% | P2=92.52%

Câu   : Chất lượng kém, vỡ sau 1 ngày, không mua lại nữa.
Nhãn  : 😠 Tiêu cực nặng (confidence: 93.45%)
Probs : N1=1.75% | N2=93.45% | P1=2.42% | P2=2.37%

Câu   : Tạm ổn, không có gì đặc biệt nhưng giá cả hợp lý.
Nhãn  : 🙂 Tích cực nhẹ (confidence: 93.98%)
Probs : N1=1.83% | N2=1.94% | P1=93.98% | P2=2.25%